# Noise Floor Inspection

This notebook helps you determine whether noise filtering is needed for your RF dataset and, if so, what threshold to use.

## What This Notebook Does

1. **Loads your data** - Reads raw RF files and computes window powers
2. **Compares detection methods** - Provides multiple statistical methods (percentile, histogram, statistical) to determine noise thresholds
3. **Shows filtering impact** - Visualizes how many windows would be filtered at different thresholds
4. **Visualizes examples** - Displays sample windows that would be kept vs. filtered to help you understand the impact
5. **Helps you decide** - Based on the visualizations and statistics, you can determine:
   - Whether noise filtering is needed at all (e.g., for continuous signals like DVB-T/LTE/WiFi, filtering may not be appropriate)
   - What threshold value to use if filtering is desired
   - Which detection method works best for your data

## Understanding Your Data

The notebook compares multiple noise detection methods and shows their filtering impact. This helps you make an informed decision:
- **For datasets with clear noise-only periods**: Filtering can remove low-power noise windows
- **For continuous signals**: The power distribution may be uniform, indicating filtering isn't needed
- **Visual inspection**: The kept vs. filtered examples help you verify that the threshold makes sense


## Setup

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import sys

# Add parent directory to path to import pipeline modules
sys.path.insert(0, str(Path.cwd().parent))

from pipeline.io import iter_windows_for_file
from pipeline.preprocessing import compute_window_power_db, detect_noise_floor_percentile


## Step 1: Configuration

**Update these parameters** to match your dataset and desired noise filtering settings.


In [ ]:
# ============================================================================
# File to analyze
# ============================================================================
# Path to raw RF data file (.bin, .mat, or .h5)
file_path = Path("../data/sub_ghz/raw/867.4/80211ah/80211ah_mcs0_chan6_g0.0dB_att3dB_freq867.4MHz_0.mat")

# On-disk dtype of a .bin file, e.g. "complex64" or "int16". A .bin carries no
# format information, and a wrong value yields noise-like powers instead of an
# error. Ignored for .mat and .h5 files.
SOURCE_DTYPE = "complex64"

# ============================================================================
# Windowing parameters (should match your dataset config)
# ============================================================================
WINDOW_SIZE = 500
HOP_SIZE = 500
PAD_LAST = True
MAX_SAMPLES = 10000  # Maximum number of windows to sample for analysis

# ============================================================================
# Noise filtering parameters
# ============================================================================
noise_floor_percentile = 1.0  # Percentile to use as noise floor (1.0 = 1st percentile)
margin_db = 6.0  # Margin in dB to add above floor for threshold


## Step 2: Load Data and Compute Window Powers


In [ ]:
# Collect window powers and store windows for later visualization
powers_db = []
windows_stored = []  # Store actual windows for visualization
window_count = 0

print(f"Analyzing: {file_path.name}")
print(f"Window size: {WINDOW_SIZE}, Hop size: {HOP_SIZE}")
print("Loading windows...")

for window, meta in iter_windows_for_file(
    path=file_path,
    window_size=WINDOW_SIZE,
    hop_size=HOP_SIZE,
    pad_last=PAD_LAST,
    dtype=np.dtype(SOURCE_DTYPE),
):
    if window_count >= MAX_SAMPLES:
        break
    power_db = compute_window_power_db(window)
    powers_db.append(power_db)
    windows_stored.append(window.copy())  # Store window for visualization
    window_count += 1
    
    if window_count % 1000 == 0:
        print(f"  Processed {window_count} windows...")

powers_db = np.array(powers_db)
windows_stored = np.array(windows_stored)  # Shape: (N_windows, 2, WINDOW_SIZE)
print(f"\nLoaded {len(powers_db)} windows")
print(f"Power range: {powers_db.min():.2f} to {powers_db.max():.2f} dB")
print(f"Mean power: {powers_db.mean():.2f} dB")
print(f"Std power: {powers_db.std():.2f} dB")


## Step 3: Calculate Threshold and Filtering Impact

This step shows:
- The dB value of the selected percentile (e.g., 1st percentile) of window powers
- The threshold value by adding the chosen margin in dB to the percentile value
- How many windows would be kept vs. filtered using this threshold


In [ ]:
# Apply percentile + margin method
percentile_margin_floor = detect_noise_floor_percentile(powers_db, percentile=noise_floor_percentile)
percentile_margin_threshold = percentile_margin_floor + margin_db

# Create keep mask
keep_mask = powers_db >= percentile_margin_threshold
kept_indices = np.where(keep_mask)[0]
filtered_count = len(powers_db) - len(kept_indices)
filter_rate = (filtered_count / len(powers_db)) * 100

# Print results
print("=" * 60)
print("NOISE FLOOR DETECTION RESULTS")
print("=" * 60)
print(f"Method: {noise_floor_percentile:.0f}th percentile + {margin_db:.0f} dB margin")
print(f"")
print(f"Noise floor:  {percentile_margin_floor:.2f} dB")
print(f"Threshold:    {percentile_margin_threshold:.2f} dB")
print(f"")
print(f"Windows kept:     {len(kept_indices):,} / {len(powers_db):,} ({(len(kept_indices)/len(powers_db)*100):.1f}%)")
print(f"Windows filtered: {filtered_count:,} / {len(powers_db):,} ({filter_rate:.1f}%)")
print("=" * 60)


## Step 4: Visualize Results

### 4.1 Power Distribution Histogram

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(powers_db, bins=100, alpha=0.7, edgecolor="black", label="Window Power Distribution")
plt.axvline(percentile_margin_floor, linestyle='--', linewidth=2, color='orange',
            label=f'Noise Floor ({noise_floor_percentile:.0f}th percentile: {percentile_margin_floor:.2f} dB)')
plt.axvline(percentile_margin_threshold, linestyle=':', linewidth=2, color='red',
            label=f'Threshold (Floor + {margin_db:.0f} dB: {percentile_margin_threshold:.2f} dB)')
plt.xlabel("Power (dB)")
plt.ylabel("Number of Windows")
plt.title(f"Window Power Distribution: {file_path.name}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4.2 Compare Multiple Detection Methods

Compare different statistical methods to estimate noise thresholds and see how they compare. This helps you understand the range of possible threshold values and choose an appropriate method for your data.


In [ ]:
# Apply all detection methods
from pipeline.preprocessing import detect_noise_floor_histogram, detect_noise_floor_statistical


methods = {
    "Percentile (5%)": detect_noise_floor_percentile(powers_db, percentile=5.0),
    "Percentile (10%)": detect_noise_floor_percentile(powers_db, percentile=10.0),
    "Histogram": detect_noise_floor_histogram(powers_db, bins=100, noise_fraction=0.1),
    "Statistical": detect_noise_floor_statistical(powers_db, noise_fraction=0.1, std_multiplier=2.0),
    f"Percentile+Margin (1%+{margin_db}dB)": percentile_margin_threshold
}

# Percentile + Margin method (from reference implementation)
# This uses a low percentile as floor, then adds a margin in dB
percentile_margin_floor = detect_noise_floor_percentile(powers_db, percentile=noise_floor_percentile)
percentile_margin_threshold = percentile_margin_floor + margin_db
methods[f"Percentile+Margin (1%+{margin_db}dB)"] = percentile_margin_threshold

# Compute max(I) for each window to see how this method would filter
max_i_per_window = np.array([np.max(np.abs(window[0])) for window in windows_stored])

print("Noise Floor Estimates (Power-based methods):")
print("-" * 50)
power_methods = {k: v for k, v in methods.items() if v is not None}
for method, threshold in power_methods.items():
    print(f"  {method:20s}: {threshold:8.2f} dB")

avg_threshold = np.mean(list(power_methods.values()))
print(f"\n  Average (power methods): {avg_threshold:8.2f} dB")
print(f"\n  Suggested threshold: {avg_threshold:.1f} dB")

### 4.3 Visualize Method Comparisons

The plots below show:
- **Top left**: Power distribution histogram with all method thresholds overlaid as vertical lines
- **Top right**: Bar chart comparing the threshold values from each method
- **Bottom left**: Filtering impact showing what percentage of windows would be filtered by each method's threshold


In [ ]:
# Visualize with all thresholds
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Power histogram with thresholds
ax1 = axes[0, 0]
ax1.hist(powers_db, bins=100, alpha=0.7, edgecolor="black")
colors = ["red", "orange", "green", "blue", "purple"]
for (name, threshold), color in zip(power_methods.items(), colors):
    ax1.axvline(
        threshold,
        color=color,
        linestyle="--",
        linewidth=2,
        label=f"{name}: {threshold:.2f} dB",
    )
ax1.set_xlabel("Power (dB)")
ax1.set_ylabel("Number of Windows")
ax1.set_title("Power Distribution with Noise Floor Estimates")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Bar chart comparing power-based methods
ax2 = axes[0, 1]
names = list(power_methods.keys())
values = list(power_methods.values())
ax2.barh(names, values, alpha=0.7)
ax2.set_xlabel("Noise Floor (dB)")
ax2.set_title("Comparison of Power-Based Detection Methods")
ax2.grid(True, alpha=0.3, axis="x")

# Plot 3: Comparison of filtering results
ax3 = axes[1, 0]
method_names = list[str](power_methods.keys())
filter_rates_power = [(np.sum(powers_db < t) / len(powers_db) * 100) for t in power_methods.values()]
filter_rates = filter_rates_power

bars = ax3.barh(method_names, filter_rates, alpha=0.7)
ax3.set_xlabel("Filter Rate (%)")
ax3.set_title("Filtering Impact Comparison")
ax3.grid(True, alpha=0.3, axis="x")
# Add value labels on bars
for bar, rate in zip(bars, filter_rates):
    width = bar.get_width()
    ax3.text(width, bar.get_y() + bar.get_height()/2, 
             f'{rate:.1f}%', ha='left', va='center', fontsize=9)

# Hide the empty subplot
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()


### 4.4 Filtering Impact Curve

This plot shows how the percentage of filtered windows changes as you vary the threshold. This helps you understand the sensitivity of filtering to threshold choice and see what filter rate you would get at different threshold values.


In [ ]:
# Test different threshold values
thresholds = np.linspace(powers_db.min(), powers_db.max(), 50)
filter_rates = []

for threshold in thresholds:
    filtered = np.sum(powers_db < threshold)
    filter_rate = (filtered / len(powers_db)) * 100
    filter_rates.append(filter_rate)

plt.figure(figsize=(12, 6))
plt.plot(thresholds, filter_rates, linewidth=2)
plt.xlabel("Noise Floor Threshold (dB)")
plt.ylabel("Percentage of Windows Filtered (%)")
plt.title("Filtering Impact vs. Threshold")
plt.grid(True, alpha=0.3)

# Mark the suggested threshold
avg_threshold = np.mean(list(methods.values()))
filter_rate_at_avg = (np.sum(powers_db < avg_threshold) / len(powers_db)) * 100
plt.axvline(avg_threshold, color="red", linestyle="--", linewidth=2, 
            label=f"Suggested ({avg_threshold:.1f} dB): {filter_rate_at_avg:.1f}% filtered")
plt.legend()
plt.tight_layout()
plt.show()

print(f"At suggested threshold ({avg_threshold:.1f} dB):")
print(f"  Windows filtered: {np.sum(powers_db < avg_threshold):,} / {len(powers_db):,}")
print(f"  Filter rate: {filter_rate_at_avg:.1f}%")


## Threshold Visualization on Sample Windows

Visualize the chosen noise threshold overlaid on randomly selected sample windows to see how it relates to actual signal power.


In [ ]:

# Randomly select 5 windows
n_samples = min(5, len(windows_stored))
sample_indices = np.random.choice(len(windows_stored), size=n_samples, replace=False)

# Convert threshold from dB to linear power for comparison
threshold_linear = 10 ** (percentile_margin_threshold / 10.0)

# Plot power over time for each sample window
fig, axes = plt.subplots(n_samples, 1, figsize=(14, 3 * n_samples))
if n_samples == 1:
    axes = [axes]

for idx, ax in zip(sample_indices, axes):
    window = windows_stored[idx]  # Shape: (2, WINDOW_SIZE)
    window_power_db = powers_db[idx]
    
    # Compute instantaneous power: I² + Q²
    instantaneous_power = window[0] ** 2 + window[1] ** 2
    
    # Time axis
    time_samples = np.arange(WINDOW_SIZE)
    
    # Plot instantaneous power
    ax.plot(time_samples, instantaneous_power, alpha=0.7, linewidth=1, label='Instantaneous Power')
    
    # Overlay threshold as horizontal line
    ax.axhline(threshold_linear, color='red', linestyle='--', linewidth=2, 
               label=f'Threshold ({percentile_margin_threshold:.1f} dB)')
    
    # Add window average power as horizontal line
    window_avg_power = np.mean(instantaneous_power)
    ax.axhline(window_avg_power, color='green', linestyle=':', linewidth=1.5,
               label=f'Window Avg Power ({window_power_db:.1f} dB)')
    
    # Shade region below threshold
    ax.axhspan(0, threshold_linear, alpha=0.15, color='red', label='Below Threshold')
    
    # Determine filtering status
    filtered = window_power_db < percentile_margin_threshold
    status = "FILTERED" if filtered else "KEPT"
    
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Power (Linear)')
    ax.set_title(f'Sample Window {idx}: Power={window_power_db:.2f} dB | {status}')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
filtered_samples = np.sum(powers_db[sample_indices] < percentile_margin_threshold)
print(f"Summary of {n_samples} random samples:")
print(f"  Windows below threshold (filtered): {filtered_samples}/{n_samples}")
print(f"  Windows above threshold (kept): {n_samples - filtered_samples}/{n_samples}")


## Example of Kept and Discarded samples

In [ ]:
# Find sample indices for kept and discarded from the randomly selected samples
# Note: sample_indices is defined in the previous cell (Step 4.3)
kept_sample_indices = [i for i in sample_indices if powers_db[i] >= percentile_margin_threshold]
discarded_sample_indices = [i for i in sample_indices if powers_db[i] < percentile_margin_threshold]

# Limit to up to 3 samples per group
kept_sample_indices = kept_sample_indices[:3]
discarded_sample_indices = discarded_sample_indices[:3]

def plot_iq_samples(indices, status_label, color):
    """Plot I and Q channels for sample windows.
    
    Args:
        indices: List of window indices to plot
        status_label: "Kept" or "Discarded" 
        color: Color for the plot (not used but kept for consistency)
    """
    n = len(indices)
    if n == 0:
        return
    fig, axs = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
    for i, idx in enumerate(indices):
        window = windows_stored[idx]  # Shape: (2, WINDOW_SIZE)
        ax = axs[0, i]
        ax.plot(window[0], label="I", color="tab:blue")
        ax.plot(window[1], label="Q", color="tab:orange")
        ax.set_title(f"{status_label} Sample {idx}\nP={powers_db[idx]:.1f} dB")
        ax.set_xlabel("Sample Index")
        ax.set_ylabel("Amplitude")
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.suptitle(f"IQ Plots for {status_label} Samples ({n} Shown)", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

if kept_sample_indices:
    plot_iq_samples(kept_sample_indices, "Kept", color="green")
else:
    print("No kept samples available for plotting.")

if discarded_sample_indices:
    plot_iq_samples(discarded_sample_indices, "Discarded", color="red")
else:
    print("No discarded samples available for plotting.")



## Configuration Snippet

Based on the visualizations above, choose the appropriate configuration option:

- **If your power distribution shows clear separation** between noise and signal windows (bimodal distribution with distinct low-power and high-power peaks), use **Option 1 or 2** to filter out low-power noise windows.
- **If your power distribution is uniform or narrow** (e.g., continuous signals like DVB-T/LTE/WiFi where all windows contain signal), use **Option 3** to disable filtering, as there are no distinct noise-only periods.
- **If filtering impact is very high** (e.g., >90% of windows would be filtered), consider whether filtering is appropriate for your use case - you may want to use **Option 3** instead.

Copy the appropriate configuration to your dataset's `config.py` file:


In [ ]:
# Generate config snippet with contextual guidance
# Analyze the data to provide recommendations
power_std = powers_db.std()
power_range = powers_db.max() - powers_db.min()
filter_rate_at_threshold = (np.sum(powers_db < percentile_margin_threshold) / len(powers_db)) * 100

# Determine if distribution suggests filtering is appropriate
# Narrow distribution (low std relative to range) suggests continuous signal
is_narrow_distribution = power_std < (power_range * 0.1)
is_high_filter_rate = filter_rate_at_threshold > 90

print("# ============================================================================")
print("# Configuration Options")
print("# ============================================================================")
print()
print(f"# Your data characteristics:")
print(f"#   Power range: {power_range:.2f} dB")
print(f"#   Power std: {power_std:.2f} dB")
print(f"#   Filter rate at threshold: {filter_rate_at_threshold:.1f}%")
print()
if is_narrow_distribution or is_high_filter_rate:
    print("# RECOMMENDATION: Your power distribution appears uniform/narrow or filtering")
    print("# would remove most windows. Consider Option 3 (disable filtering).")
else:
    print("# RECOMMENDATION: Your power distribution shows variation that may benefit")
    print("# from filtering. Consider Option 1 or 2.")
print()
print("# ============================================================================")
print("# Option 1: Use the detected threshold manually")
print("# ============================================================================")
print("# Use this if you want a fixed threshold across all files.")
print("# Best for: Consistent noise characteristics across files")
print()
print(f'NOISE_POWER_THRESHOLD_DB = {percentile_margin_threshold:.1f}  # Detected: {noise_floor_percentile:.0f}th percentile + {margin_db:.0f} dB margin')
print()
print("# ============================================================================")
print("# Option 2: Use auto-detection with percentile + margin method")
print("# ============================================================================")
print("# Use this if noise characteristics vary per file.")
print("# Best for: Datasets where noise floor differs between files")
print()
print('NOISE_POWER_THRESHOLD_DB = "auto"')
print('NOISE_FLOOR_DETECTION_METHOD = "percentile"')
print(f'NOISE_FLOOR_DETECTION_PARAMS = {{')
print(f'    "percentile": {noise_floor_percentile:.1f},  # Percentile for noise floor')
print(f'    "margin_db": {margin_db:.1f},  # Margin to add above floor')
print(f'}}')
print()
print("# ============================================================================")
print("# Option 3: Disable filtering")
print("# ============================================================================")
print("# Use this if your signals are continuous (no distinct noise-only periods).")
print("# Best for: Continuous signals like DVB-T, LTE, WiFi where all windows contain signal")
print()
print("NOISE_POWER_THRESHOLD_DB = None")
